<a href="https://colab.research.google.com/github/KurniaYufi/sentiment-analysis-kematian-ali-khamenei/blob/main/scraping/kompas/kompas_content_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisis Artikel Kompas: Scraping dan Persiapan Data

In [1]:
!pip install beautifulsoup4 requests pandas lxml

!pip uninstall icu -y
!pip uninstall polyglot -y
!pip uninstall pyicu -y

!pip install polyglot pycld2 pyicu morfessor
!pip install deep-translator
!pip install Sastrawi
!pip install stanza
!pip install nltk
!pip install wordcloud

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.3/126.3 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.2/268.2 kB 10.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 79.1 MB/s eta 0:00:00
  Created wheel for polyglot: filename=polyglot-16.7.4-py2.py3-none-any.whl size=52563 sha256=95879aa3000162efbf43933a580a1747c2ae12f1be7e920edd6ebb50e9d32c19
  Stored in directory: /root/.cache/pip/wheels/c7/5e/28/47349211ec1f91379f41ed10bc2520f7071ecfb6cbe182f6fe
  Created wheel for pyicu: filename=pyicu-2.16.2-cp312-cp312-linux_x86_64.whl size=2720236 sha256=ed87525c3359e5458d38a350b22d93a136d3032a36be94e8df62d5f1c46cc5f8
  Stored in directory: /root/.cache/pip/wheels/25/f3/cd/4923c874cedf8cdb8608035f48bb726fa040a98a66e2b13cea
Successfully built polyglot pyicu
   ━━━━━━━━━━━━

## 1. Instalasi dan Persiapan Environment

Bagian ini menginstal semua library Python yang diperlukan untuk scraping web, pemrosesan teks, dan analisis data.

In [2]:
import io
import string
import re
import time
from collections import defaultdict, Counter
from urllib.parse import urlparse

import pandas as pd
import numpy as np

import requests
from bs4 import BeautifulSoup
from google.colab import files

import nltk
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

import stanza

from textblob import TextBlob
from deep_translator import GoogleTranslator

import matplotlib.pyplot as plt
from wordcloud import WordCloud

## 2. Impor Library

Memuat semua modul dan library yang akan digunakan dalam proyek ini, termasuk `pandas` untuk data, `requests` dan `BeautifulSoup` untuk web scraping, `nltk` untuk pemrosesan bahasa alami, dan `matplotlib` untuk visualisasi.

In [3]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## 3. Unduh Sumber Daya NLTK

`nltk` (Natural Language Toolkit) memerlukan sumber daya seperti `stopwords` (kata-kata umum yang sering dihapus) dan `punkt` (untuk tokenisasi teks) untuk berfungsi dengan baik.

In [6]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

Saving kompas_khamenei_articles.csv to kompas_khamenei_articles.csv


## 4. Unggah File Daftar URL

Bagian ini meminta pengguna untuk mengunggah file CSV yang berisi daftar URL artikel Kompas yang akan di-scrape. File akan dibaca dan URL akan diekstrak.

In [7]:
try:
    links_df = pd.read_csv(io.BytesIO(uploaded[filename]))
    if 'Link' in links_df.columns:
        urls = links_df['Link'].tolist()
        print(f"Berhasil memuat {len(urls)} URL dari '{filename}'.")
        print('\nContoh URL:')
        for u in urls[:5]:
            print(' ', u)
    else:
        print("Error: Kolom 'Link' tidak ditemukan dalam CSV.")
        urls = []
except Exception as e:
    print(f'Terjadi kesalahan: {e}')
    urls = []

Berhasil memuat 100 URL dari 'kompas_khamenei_articles.csv'.

Contoh URL:
  https://www.kompas.com/global/read/2026/03/02/065000070/setelah-ali-khamenei-pergi
  https://cahaya.kompas.com/aktual/26C14141821890/profil-mojtaba-khamenei-pemimpin-tertinggi-iran-pengganti-ayatollah-ali-khamenei
  https://nasional.kompas.com/read/2026/03/09/04000071/megawati-ali-khamenei-dan-keadilan-dunia
  https://www.kompas.com/cekfakta/read/2026/03/12/192000582/-hoaks-foto-gibran-pernah-mengunjungi-ali-khamenei
  https://www.kompas.com/cekfakta/read/2026/03/02/170700882/-hoaks-foto-jenazah-ali-khamenei-di-reruntuhan


## 5. Memuat URL dari CSV

Kode ini membaca file CSV yang diunggah dan mengekstrak kolom 'Link' untuk mendapatkan daftar URL artikel yang akan diproses. Ini juga mencetak beberapa URL contoh untuk verifikasi.

In [8]:
def get_article_type(url):
    # Menentukan tipe artikel Kompas berdasarkan subdomain URL.
    parsed = urlparse(url)
    domain = parsed.netloc.lower()
    if 'cahaya.kompas.com' in domain:
        return 'cahaya'
    elif 'video.kompas.com' in domain:
        return 'video'
    elif 'english.kompas.com' in domain:
        return 'english'
    else:
        return 'standard'  # www, internasional, nasional, regional, bola


def scrape_kompas(url):
    # Scraping utama untuk semua subdomain Kompas.
    # Setiap tipe subdomain memiliki selector HTML yang berbeda.
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/124.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://www.google.com/',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
    }

    try:
        response = requests.get(url, headers=headers, timeout=20)

        if response.status_code != 200:
            return {
                'url': url, 'title': None, 'content': None,
                'article_type': get_article_type(url),
                'error': f'HTTP {response.status_code}'
            }

        soup = BeautifulSoup(response.content, 'lxml')
        article_type = get_article_type(url)

        # --- Ekstrak judul ---
        # Prioritas: og:title > h1.read__title > h1 umum > <title>
        title = None
        og_title = soup.find('meta', property='og:title')
        if og_title and og_title.get('content'):
            title = og_title['content'].strip()
        if not title:
            h1 = (soup.find('h1', class_='read__title') or
                  soup.find('h1', class_='article__title') or
                  soup.find('h1'))
            if h1:
                title = h1.get_text(strip=True)
        if not title:
            tag_title = soup.find('title')
            if tag_title:
                title = tag_title.get_text(strip=True)

        # --- Ekstrak konten berdasarkan tipe subdomain ---
        content = None
        paragraphs = []

        if article_type == 'cahaya':
            # cahaya.kompas.com - format jurnalisme warga/aktual
            # Selector utama: div.col-bs9-inner atau div[data-content]
            content_div = (
                soup.find('div', class_='col-bs9-inner') or
                soup.find('div', attrs={'data-content': True}) or
                soup.find('div', class_='article__body') or
                soup.find('article')
            )
            if content_div:
                for el in content_div.find_all(['script', 'style', 'figure', 'aside']):
                    el.decompose()
                for el in content_div.find_all(
                    class_=re.compile(r'ads|iklan|caption|photo|related|widget', re.I)
                ):
                    el.decompose()
                paragraphs = content_div.find_all('p')
            else:
                paragraphs = soup.find_all('p')

        elif article_type == 'video':
            # video.kompas.com - konten berupa deskripsi video
            content_div = (
                soup.find('div', class_='video__description') or
                soup.find('div', class_='video__body') or
                soup.find('div', class_='detail__body')
            )
            if content_div:
                paragraphs = content_div.find_all('p')
            # Fallback: meta description (jika halaman video minim teks)
            if not paragraphs:
                meta_desc = (soup.find('meta', attrs={'name': 'description'}) or
                             soup.find('meta', property='og:description'))
                if meta_desc and meta_desc.get('content'):
                    content = meta_desc['content'].strip()

        elif article_type == 'english':
            # english.kompas.com - format lama (/read/xml/) atau baru
            content_div = (
                soup.find('div', class_='read__content') or
                soup.find('div', id='content') or
                soup.find('div', class_='article-content') or
                soup.find('article')
            )
            if content_div:
                for el in content_div.find_all(['script', 'style', 'figure', 'aside']):
                    el.decompose()
                paragraphs = content_div.find_all('p')
            else:
                paragraphs = soup.find_all('p')

        else:
            # standard: www, internasional, nasional, regional, bola.kompas.com
            # Selector utama Kompas standar: div.read__content
            content_div = (
                soup.find('div', class_='read__content') or
                soup.find('div', class_='article__content') or
                soup.find('div', class_='detail__body-text') or
                soup.find('article') or
                soup.find('div', class_='content')
            )
            if content_div:
                # Hapus elemen non-artikel: iklan, foto, berita terkait
                for el in content_div.find_all(['script', 'style', 'figure', 'aside']):
                    el.decompose()
                for el in content_div.find_all(
                    class_=re.compile(
                        r'ads|iklan|photo|caption|related|recommendation|widget|author|tag',
                        re.I
                    )
                ):
                    el.decompose()
                paragraphs = content_div.find_all('p')
            else:
                # Fallback jika selector utama tidak ditemukan
                paragraphs = soup.find_all('p')

        # Gabungkan paragraf menjadi teks
        if paragraphs and not content:
            content = '\n'.join([
                p.get_text(separator=' ', strip=True)
                for p in paragraphs if p.get_text(strip=True)
            ])

        # Bersihkan newline berlebih
        if content:
            content = re.sub(r'\n{3,}', '\n\n', content).strip()

        return {
            'url': url,
            'title': title,
            'content': content,
            'article_type': article_type,
            'error': None
        }

    except Exception as e:
        return {
            'url': url, 'title': None, 'content': None,
            'article_type': get_article_type(url),
            'error': str(e)
        }

## 6. Fungsi Scraping Artikel Kompas

Fungsi `get_article_type` dan `scrape_kompas` dirancang untuk mengambil konten artikel dari berbagai subdomain Kompas (`www`, `cahaya`, `video`, `english`). Fungsi ini mengidentifikasi struktur HTML yang berbeda untuk setiap tipe artikel dan mengekstrak judul serta konten utama.

In [9]:
# Jalankan scraping untuk semua URL
data = []
if urls:
    print(f'Memulai scraping {len(urls)} artikel...\n')
    for i, url in enumerate(urls):
        result = scrape_kompas(url)
        data.append(result)
        if (i + 1) % 10 == 0 or (i + 1) == len(urls):
            sukses = sum(1 for d in data if d['content'])
            print(f'[{i+1}/{len(urls)}] Berhasil: {sukses} | Gagal: {i+1 - sukses}')
        time.sleep(0.5)  # Jeda untuk menghindari rate-limiting
else:
    print('Tidak ada URL untuk di-scrape.')

df_article = pd.DataFrame(data)
print('\nHasil scraping:')
print(df_article[['url', 'title', 'article_type', 'error']].head(10))
total = len(df_article)
berhasil = df_article['content'].notna().sum()
print(f'\nTotal: {total} | Berhasil: {berhasil} | Gagal: {total - berhasil}')

Memulai scraping 100 artikel...

[10/100] Berhasil: 10 | Gagal: 0
[20/100] Berhasil: 20 | Gagal: 0
[30/100] Berhasil: 30 | Gagal: 0
[40/100] Berhasil: 40 | Gagal: 0
[50/100] Berhasil: 50 | Gagal: 0
[60/100] Berhasil: 60 | Gagal: 0
[70/100] Berhasil: 70 | Gagal: 0
[80/100] Berhasil: 80 | Gagal: 0
[90/100] Berhasil: 90 | Gagal: 0
[100/100] Berhasil: 96 | Gagal: 4

Hasil scraping:
                                                 url  \
0  https://www.kompas.com/global/read/2026/03/02/...   
1  https://cahaya.kompas.com/aktual/26C1414182189...   
2  https://nasional.kompas.com/read/2026/03/09/04...   
3  https://www.kompas.com/cekfakta/read/2026/03/1...   
4  https://www.kompas.com/cekfakta/read/2026/03/0...   
5  https://www.kompas.com/global/read/2026/03/22/...   
6  https://www.kompas.com/tren/read/2026/03/09/08...   
7  https://www.kompas.com/tren/read/2026/03/09/16...   
8  https://nasional.kompas.com/read/2026/03/04/21...   
9  https://www.kompas.com/global/read/2025/02/07/...   

  

## 7. Jalankan Proses Scraping

Bagian ini mengiterasi melalui daftar URL yang telah dimuat, memanggil fungsi `scrape_kompas` untuk setiap URL, dan menyimpan hasilnya ke dalam list `data`. Proses ini juga mencetak progres dan ringkasan keberhasilan/kegagalan scraping.

In [10]:
print('Distribusi tipe artikel:')
print(df_article['article_type'].value_counts())

df_article.to_csv('content_articles_kompas.csv', index=False)
print("\nScraping mentah disimpan ke 'scraped_articles_kompas.csv'")

Distribusi tipe artikel:
article_type
standard    83
video       10
english      4
cahaya       3
Name: count, dtype: int64

Scraping mentah disimpan ke 'scraped_articles_kompas.csv'


## 8. Analisis Hasil Scraping dan Penyimpanan Data

Setelah scraping selesai, kode ini menampilkan distribusi tipe artikel yang ditemukan dan menyimpan semua data yang telah di-scrape ke dalam file CSV baru bernama `content_articles_kompas.csv`.